# PathSnap — Interactive Plots & Folium Maps (In-Notebook Viewer)
Run these cells to render all **M2 Model Comparisons**, **M3 HMM Ablation & Per-Trip Plots**, and **Interactive 5-Layer Folium Maps** directly inside the notebook without saving or creating any new files on disk.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium
from IPython.display import display

REPO_ROOT = Path.cwd().parent if Path.cwd().name in ("m3_hmm", "m2_models") else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

%matplotlib inline
print(f"Repository root: {REPO_ROOT}")

## 1. M2 Candidate Classification Results & Comparison Plots (Inline)

In [ ]:
df_m2_cv = pd.read_csv(REPO_ROOT / "m2_models/results/cv_results.csv")
df_m2_test = pd.read_csv(REPO_ROOT / "m2_models/results/student_test_results.csv")
display(df_m2_test)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), dpi=130)

d1 = df_m2_test.sort_values("top1_accuracy", ascending=True)
axes[0].barh(d1["model"], d1["top1_accuracy"] * 100, color="#1976d2", edgecolor="black")
axes[0].set_xlim(84, 91)
axes[0].set_title("M2 Student Test: Top-1 Accuracy (%)", fontweight="bold")

d2 = df_m2_test.sort_values("macro_f1", ascending=True)
axes[1].barh(d2["model"], d2["macro_f1"], color="#388e3c", edgecolor="black")
axes[1].set_xlim(0.84, 0.95)
axes[1].set_title("M2 Student Test: Macro F1-Score", fontweight="bold")

d3 = df_m2_test.sort_values("log_loss", ascending=False)
axes[2].barh(d3["model"], d3["log_loss"], color="#d32f2f", edgecolor="black")
axes[2].set_title("M2 Student Test: Log Loss (Lower is Better)", fontweight="bold")

plt.tight_layout()
plt.show()

## 2. M3 HMM + Viterbi Ablation & Per-Trip Results (Inline)

In [ ]:
df_m3_ablation = pd.read_csv(REPO_ROOT / "m3_hmm/outputs/hmm_ablation_comparison.csv")
df_m3_err = pd.read_csv(REPO_ROOT / "m3_hmm/outputs/error_analysis_summary.csv", dtype={"trip_id": str})
display(df_m3_ablation)
display(df_m3_err)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), dpi=130)
short_names = ["Clf Only", "HMM 1:\nTopo", "HMM 2:\nKinematic", "HMM 3:\nEmpirical", "HMM:\nMaj Vote", "HMM:\n3-Trans Ens", "HMM:\nFull Ens"]
colors = ["#8c8c8c", "#4c78a8", "#72b7b2", "#f58518", "#e45756", "#54a24b", "#2ca02c"]

for ax, (col, title) in zip(axes, [
    ("top1_accuracy_all_weighted", "Observation Top-1 Accuracy (%)"),
    ("route_continuity_mean", "Reconstructed Route Continuity (%)"),
    ("route_levenshtein_sim_mean", "Levenshtein Sequence Similarity (%)"),
]):
    vals = df_m3_ablation[col].to_numpy() * 100.0
    bars = ax.bar(short_names, vals, color=colors, edgecolor="#222")
    ax.set_title(title, fontweight="bold")
    ax.set_ylim(max(0, vals.min() - 6), min(102, vals.max() + 4))
    ax.tick_params(axis="x", labelsize=8.5)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.4, f"{v:.2f}%", ha="center", fontsize=8, fontweight="bold")

plt.tight_layout()
plt.show()

## 3. Interactive Folium Map Viewer (Renders Inline Without Saving Files)
Change `TRIP_ID` to any of the 10 `student_test` trips (`"00000000"`, `"00000007"`, `"00000009"`, `"00000040"`, `"00000052"`, `"00000056"`, `"00000065"`, `"00000076"`, `"00000077"`, `"00000091"`).

In [ ]:
from m1_data.loader import load_trip, load_route
from m1_data.network import RoadNetwork
from m1_data.features import build_trip_table
from m1_data.maps import trip_map, add_route, add_legend
from m3_hmm.transitions import TripGraph, HMMParams, EmpiricalPriors
from m3_hmm.viterbi import decode_trip_all_methods
from m3_hmm.model_bridge import load_m2_models, predict_m2_probabilities_on_df

def show_trip_map_inline(trip_id: str = "00000007", partition: str = "student_test"):
    trip = load_trip(partition, trip_id)
    ref_route = load_route(partition, trip_id)
    net = RoadNetwork(trip)
    px, py = net.proj.to_xy(trip.lon, trip.lat)
    graph = TripGraph(net.arc_from, net.arc_to, net.arc_len)

    df_trip, _ = build_trip_table(trip, route=ref_route)
    df_trip = predict_m2_probabilities_on_df(df_trip, load_m2_models())

    params = HMMParams.load(REPO_ROOT / "m3_hmm/artifacts/hmm_config.json")
    priors = EmpiricalPriors.load(REPO_ROOT / "m3_hmm/artifacts/empirical_priors.npy")
    decoded = decode_trip_all_methods(df_trip, graph, params, priors, gps_xy=(px, py))

    clf_res = decoded["Classifier-Only (No HMM)"]
    hmm_res = decoded["HMM Ensemble (3-Transition Soft Average)"]

    routes = [(ref_route, "Reference Route (Ground Truth)", "#2ca02c")] if ref_route is not None else []
    m = trip_map(trip, net, network_radius_m=80.0, routes=routes)
    add_route(m, net, clf_res["route_arcs"], name="Classifier-Only Route (No HMM)", color="#ff7f0e", weight=4, dash="6, 6")
    add_route(m, net, hmm_res["route_arcs"], name="HMM + Viterbi Ensemble Route", color="#d62728", weight=5)
    add_legend(
        m,
        entries=[
            ("GPS Observations (.track)", "#1f77b4"),
            ("Road Network (.arcs within 80m)", "#9e9e9e"),
            ("Reference Route (.route)", "#2ca02c"),
            ("Classifier-Only Route", "#ff7f0e"),
            ("HMM + Viterbi Route", "#d62728"),
        ],
        title=f"PathSnap Trip {trip_id} ({partition})",
    )
    folium.LayerControl(collapsed=False).add_to(m)
    return m

# Renders directly inside the notebook cell without creating any HTML file on disk:
show_trip_map_inline("00000007")